# DSA4213 Assignment 1 — Part III: Downstream Sentiment Classification (Colab version)

Frozen LMs from Parts I–II → fixed-length review representations → the same logistic-regression classifier for every representation.

* **Cell 0:** project folder setup + GPU check
* **Cell 0b:** build the Rotten Tomatoes splits from `data/raw/` and fetch the Part I–II checkpoints if missing.
* **Cell 1:** paths and config (same project folder as Parts I–II).
* **Cell 2:** tokenize the Rotten Tomatoes splits with the Part I tokenizer and the vocabulary stored in `results/lstm.pt`.
* **Cell 3:** Part I model classes, shared feature extraction (last-token and mean pooling), LSTM.
* **Cell 4:** Part II LSTM embedding variants (3 settings × 3 seeds).
* **Cell 5:** classifier (C tuned on validation) → test results table + TF-IDF reference.
* **Cell 6:** error analysis (subsets, McNemar test, example reviews).

**Requires:** nothing beyond the repo. Cell 0b rebuilds `data/raw/rotten_tomatoes_critic_reviews.csv` from the committed `data/raw/rt_critic_reviews.csv.gz.part*` files, builds the `data/prepared/rt_*.csv` splits from it (by running `Part3_data_prep_RottenTomatoes.ipynb`) and downloads the Part I–II checkpoints (`results/lstm.pt`, `results/part2/`) if they are missing.

**Attribution:** LSTM / positional-encoding classes adapted from the DSA4213 Lecture 3 notebook (via Parts I–II).

In [2]:
# ==========================================
# CELL 0: SETUP 
# ==========================================
import os, sys, gzip, subprocess, zipfile, urllib.request
from pathlib import Path
import torch

REPO_URL  = "https://github.com/LeuRNburgJoe11/DSA4213_Assignment.git"
REPO_NAME = "DSA4213_Assignment"
CKPT_URL  = ("https://github.com/LeuRNburgJoe11/DSA4213_Assignment/"
             "releases/download/v1.0/checkpoints.zip")

IN_COLAB = "google.colab" in sys.modules

def _is_repo_root(p: Path) -> bool:
    return (p / "README.md").exists() and (p / "data" / "prepare_data.py").exists()

if IN_COLAB:
    PROJECT_DIR = Path("/content") / REPO_NAME
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    here = Path.cwd().resolve()
    PROJECT_DIR = next((p for p in [here, *here.parents] if _is_repo_root(p)), here)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
(PROJECT_DIR / "results" / "part2").mkdir(parents=True, exist_ok=True)

def require(path, hint=""):
    """Fail early with a clear message instead of deep inside a later cell."""
    if not (PROJECT_DIR / path).exists():
        raise FileNotFoundError(f"Missing {path}. {hint}")

def ensure_wikitext():
    """Rebuild data/processed/{train,valid,test}.txt if this runtime doesn't have them."""
    if not all((PROJECT_DIR / "data/processed" / f"{s}.txt").exists()
               for s in ("train", "valid", "test")):
        subprocess.run([sys.executable, "data/prepare_data.py"], check=True)

def ensure_rt_raw():
    """Rebuild data/raw/rotten_tomatoes_critic_reviews.csv from the 3 gzip parts in the repo
    (only the columns the prep notebook reads, split to stay under GitHub's 25 MB limit)."""
    rt_csv = PROJECT_DIR / "data/raw/rotten_tomatoes_critic_reviews.csv"
    parts = sorted((PROJECT_DIR / "data/raw").glob("rt_critic_reviews.csv.gz.part*"))
    if rt_csv.exists() or not parts:
        return
    rt_csv.write_bytes(gzip.decompress(b"".join(p.read_bytes() for p in parts)))

def ensure_rt_prepared():
    """Build data/prepared/rt_*.csv by executing the data-prep notebook in this runtime."""
    if all((PROJECT_DIR / "data/prepared" / f"rt_{s}.csv").exists()
           for s in ("train", "valid", "test")):
        return
    ensure_wikitext()
    ensure_rt_raw()
    require("data/raw/rotten_tomatoes_critic_reviews.csv",
            "Download it from Kaggle (see README > Data access) and upload it to data/raw/.")
    subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute",
                    "Part3_data_prep_RottenTomatoes.ipynb",
                    "--output", "/tmp/part3_prep_executed.ipynb"], check=True)

def ensure_checkpoints():
    """Fetch the trained LSTM checkpoints (Parts I-II) from the GitHub release."""
    have_p1 = (PROJECT_DIR / "results/lstm.pt").exists()
    have_p2 = len(list((PROJECT_DIR / "results/part2").glob("lstm_*.pt"))) >= 12
    if have_p1 and have_p2:
        return
    zpath = PROJECT_DIR / "checkpoints.zip"
    print("Downloading checkpoints from GitHub release ...")
    urllib.request.urlretrieve(CKPT_URL, zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(PROJECT_DIR)  # contains results/lstm.pt and results/part2/lstm_*.pt
    zpath.unlink()

print("Working directory:", Path.cwd())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU available. In Colab, select: Runtime > Change runtime type > T4 GPU")

Working directory: /content/DSA4213_Assignment
GPU: Tesla T4


In [ ]:
# ==========================================
# CELL 0a: CHECKPOINTS FROM DRIVE (shortcut until checkpoints.zip is on the v1.0 release)
# ==========================================
from google.colab import drive
drive.mount("/content/drive")
ZIP = "/content/drive/MyDrive/dsa4213/ind_assignment/checkpoints.zip"
with zipfile.ZipFile(ZIP) as z:
    z.extractall(PROJECT_DIR)   # -> results/lstm.pt, results/part2/...
print(sorted(p.name for p in (PROJECT_DIR / "results/part2").glob("lstm_*.pt")))

In [4]:
# ==========================================
# CELL 0b: ENSURE DATA ACESS AND CHECKPOINTS
# ==========================================
ensure_rt_prepared(); 
ensure_checkpoints()

FileNotFoundError: Missing data/raw/rotten_tomatoes_critic_reviews.csv. Download it from Kaggle (see README > Data access) and upload it to data/raw/.

In [ ]:
# ==========================================
# CELL 1: PATHS & CONFIG (same project folder as Parts I–II)
# ==========================================
import os, sys, re, math, json, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# PROJECT_DIR and the working directory are set in Cell 0.

# ---------- ADAPT: where your cleaned Rotten Tomatoes splits live ----------
DOWNSTREAM_DIR = Path("data/prepared")            # Adjusted to your data path
TEXT_COL, LABEL_COL = "text", "label"             # Updated TEXT_COL to match your CSV structure
# ---------------------------------------------------------------------------

PART1_DIR = Path("results")          # lstm.pt   (Part I, Cell 6)
PART2_DIR = Path("results/part2")    # lstm_{setting}_s{seed}.pt         (Part II, Cell 7)
OUT_DIR   = Path("results/part3")

TOK_DIR, FEAT_DIR = OUT_DIR / "tokenized", OUT_DIR / "features"
for d in (TOK_DIR, FEAT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]
MAX_LEN = 32        # = sequence_length used to train every LM
BATCH_SIZE = 256
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

dfs = {}
for s in SPLITS:
    # Map 'val' split to 'rt_valid.csv' and other splits to 'rt_{split}.csv'
    filename = "rt_valid.csv" if s == "val" else f"rt_{s}.csv"
    path = DOWNSTREAM_DIR / filename
    assert path.exists(), f"Missing {path}"
    df = pd.read_csv(path)
    if LABEL_COL not in df.columns:              # build labels from the Kaggle column if needed
        df[LABEL_COL] = (df["review_type"] == "Fresh").astype(int)
    df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
    dfs[s] = df
    print(f"{s:5s}: {len(df):6d} reviews | positive share = {df[LABEL_COL].mean():.3f}")
print("Device:", DEVICE)

AssertionError: Missing data/prepared/rt_train.csv

In [10]:
# ==========================================
# CELL 2: TOKENIZE THE SENTIMENT SPLITS WITH THE PART I TOKENIZER + VOCAB
# ==========================================
# Tokenizer copied verbatim from Part I Cell 2; the vocabulary is read from the Part I
# checkpoint itself, so it is guaranteed to be the one the LMs were trained with.
START_TOKEN, END_TOKEN = "<s>", "</s>"
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"
TOKEN_PATTERN = re.compile(r"\w+(?:'\w+)?|[^\w\s]")
WIKITEXT_MARKERS = re.compile(r" @([-,.])@ ")

def clean_wikitext(line):
    return WIKITEXT_MARKERS.sub(r"\1", line)

def tokenize(text):
    normalized = clean_wikitext(text).lower().replace("’", "'")
    return TOKEN_PATTERN.findall(normalized)

ref_ckpt = torch.load(PART1_DIR / "lstm.pt", map_location="cpu", weights_only=False)
token_to_id, vocabulary = ref_ckpt["token_to_id"], ref_ckpt["vocabulary"]
PAD_ID, UNK_ID, BOS_ID = token_to_id[PAD_TOKEN], token_to_id[UNK_TOKEN], token_to_id[START_TOKEN]
print(f"Vocab size: {len(vocabulary):,} (from {PART1_DIR / 'lstm.pt'})")

def encode(text):
    """<s> + review tokens, like a WikiText line (and like the generation prompts).
    No </s>, so the 'last' state sits on the review's final real token."""
    toks = tokenize(text)
    ids = [BOS_ID] + [token_to_id.get(t, UNK_ID) for t in toks]
    return ids, len(toks), sum(t not in token_to_id for t in toks)

stats = []
for s in SPLITS:
    df = dfs[s]
    encoded = [encode(t) for t in df[TEXT_COL].astype(str)]
    keep = [i for i, (_, n, _) in enumerate(encoded) if n > 0]       # drop reviews with no tokens
    df = df.iloc[keep].reset_index(drop=True)
    encoded = [encoded[i] for i in keep]
    dfs[s] = df
    df.to_csv(TOK_DIR / f"{s}_kept.csv", index=False)   # row i here = row i of every feature matrix

    N = len(encoded)
    ids = torch.full((N, MAX_LEN), PAD_ID, dtype=torch.long)       # right padding
    lengths = torch.zeros(N, dtype=torch.long)
    full_lens, n_tok, n_oov = [], 0, 0
    for i, (seq, n, o) in enumerate(encoded):
        full_lens.append(len(seq))
        seq = seq[:MAX_LEN]                                          # keep the first MAX_LEN tokens
        ids[i, : len(seq)] = torch.tensor(seq)
        lengths[i] = len(seq)
        n_tok += n; n_oov += o
    labels = torch.tensor(df[LABEL_COL].values, dtype=torch.long)
    torch.save({"ids": ids, "lengths": lengths, "labels": labels}, TOK_DIR / f"{s}.pt")

    full_lens = np.array(full_lens)
    stats.append({"split": s, "n": N,
                  "oov_rate_%": round(100 * n_oov / n_tok, 2),
                  "truncated_%": round(100 * (full_lens > MAX_LEN).mean(), 2),
                  "median_len": int(np.median(full_lens)),
                  "p95_len": int(np.percentile(full_lens, 95))})

stats_df = pd.DataFrame(stats)
stats_df.to_csv(OUT_DIR / "tokenization_stats.csv", index=False)
print(stats_df.to_string(index=False))
# Lengths include <s>. Report OOV and truncation rates; truncation is the price of MAX_LEN = 32.

FileNotFoundError: [Errno 2] No such file or directory: 'results/lstm.pt'

In [ ]:
# Shortest review kept in each split (in Part I tokens and in whitespace words)
for s in SPLITS:
    texts = pd.read_csv(TOK_DIR / f"{s}_kept.csv")[TEXT_COL].astype(str)
    n_tok = texts.map(lambda t: len(tokenize(t)))
    n_word = texts.str.split().str.len()
    print(f"{s:5s}: min {n_tok.min()} tokens / {n_word.min()} words | "
          f"reviews with <= 3 tokens: {(n_tok <= 3).sum()}")

train: min 4 tokens / 1 words | reviews with <= 3 tokens: 0
val  : min 4 tokens / 2 words | reviews with <= 3 tokens: 0
test : min 4 tokens / 2 words | reviews with <= 3 tokens: 0


In [ ]:
# ==========================================
# CELL 3: PART I MODEL CLASSES + SHARED FEATURE EXTRACTION + LSTM
# ==========================================
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score

class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.0, tie_weights=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(hidden_dim, embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)
        if tie_weights:
            self.fc.weight = self.embedding.weight
    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.drop(self.embedding(x)), hidden)
        return self.fc(self.proj(self.drop(out))), hidden

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# ---- hidden states = final-layer states [B, T, H], i.e. everything before the LM head ----
def lstm_hidden(m, x):
    return m.lstm(m.drop(m.embedding(x)))[0]                      # [B, T, hidden_dim=256]

def recurrent_head(m, h):   return m.fc(m.proj(m.drop(h)))

def freeze(model):
    model.to(DEVICE).eval()                      # eval() turns all dropout off
    for p in model.parameters():
        p.requires_grad_(False)
    return model

def load_part1(name):
    ckpt = torch.load(PART1_DIR / f"{name}.pt", map_location="cpu", weights_only=False)
    assert ckpt["token_to_id"] == token_to_id, f"{name}.pt was trained with a different vocabulary"
    cfg, V = ckpt["config"], len(vocabulary)
    model = LSTMLanguageModel(
        vocab_size=V,
        embed_dim=cfg["embed_dim"],
        hidden_dim=cfg["hidden_dim"],
        num_layers=cfg.get("num_layers", 1),
        dropout=cfg.get("dropout", 0.0),
        tie_weights=cfg.get("tie_weights", True)
    )
    model.load_state_dict(ckpt["model_state_dict"])
    return freeze(model)

# ---- shared extraction: identical for every model ----
@torch.no_grad()
def extract_features(model, hidden_fn, split):
    """Two fixed-length representations per review:
       last = hidden state at the last real (non-pad) token
       mean = average of hidden states over real tokens (<s> included, padding excluded)
    Right padding + causal models => states at real positions never see the padding."""
    d = torch.load(TOK_DIR / f"{split}.pt")
    out = {"last": [], "mean": []}
    for i in range(0, len(d["ids"]), BATCH_SIZE):
        x = d["ids"][i : i + BATCH_SIZE].to(DEVICE)
        L = d["lengths"][i : i + BATCH_SIZE].to(DEVICE)
        h = hidden_fn(model, x).float()
        out["last"].append(h[torch.arange(len(x), device=DEVICE), L - 1])
        mask = (torch.arange(x.size(1), device=DEVICE)[None, :] < L[:, None]).unsqueeze(-1).float()
        out["mean"].append((h * mask).sum(1) / mask.sum(1))
    return {k: torch.cat(v).cpu().numpy() for k, v in out.items()}, d["labels"].numpy()

@torch.no_grad()
def sanity_checks(model, hidden_fn, head_fn, n=8):
    d = torch.load(TOK_DIR / "val.pt")
    x, L = d["ids"][:n].to(DEVICE), d["lengths"][:n].to(DEVICE)
    h = hidden_fn(model, x)
    # 1) hidden_fn + head must reproduce the model's own logits
    assert torch.allclose(head_fn(model, h), model(x)[0], atol=1e-4), "hidden_fn differs from forward()"
    # 2) a review encoded alone (no padding) must give the same 'last' state as inside the padded batch
    for j in range(n):
        alone = hidden_fn(model, x[j : j + 1, : L[j]])[0, -1]
        assert torch.allclose(h[j, L[j] - 1], alone, atol=1e-4), f"padding leaks into review {j}"

def run_extraction(name, model, hidden_fn, head_fn):
    sanity_checks(model, hidden_fn, head_fn)
    for s in SPLITS:
        feats, y = extract_features(model, hidden_fn, s)
        for pool, X in feats.items():
            np.save(FEAT_DIR / f"{name}_{pool}_{s}.npy", X)
        np.save(FEAT_DIR / f"labels_{s}.npy", y)
    print(f"[{name}] checks passed | features {feats['mean'].shape[1]}-d saved for {SPLITS}")

def quick_check(name, verbose=True):
    """Sanity check on VALIDATION only; the test set stays untouched until the final evaluation."""
    ytr, yva = np.load(FEAT_DIR / "labels_train.npy"), np.load(FEAT_DIR / "labels_val.npy")
    rows = []
    for pool in ["last", "mean"]:
        Xtr, Xva = (np.load(FEAT_DIR / f"{name}_{pool}_{s}.npy") for s in ("train", "val"))
        clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)).fit(Xtr, ytr)
        pred = clf.predict(Xva)
        rows.append({"model": name, "pool": pool, "val_acc": accuracy_score(yva, pred),
                     "val_macro_f1": f1_score(yva, pred, average="macro")})
    if verbose:
        print(f"majority-class val acc = {max(yva.mean(), 1 - yva.mean()):.3f}")
        print(pd.DataFrame(rows).round(3).to_string(index=False))
    return rows

set_seed()
lstm = load_part1("lstm")
run_extraction("lstm", lstm, lstm_hidden, recurrent_head)
quick_check("lstm")
# Both pooling methods should clearly beat the majority baseline. C=1.0 here is only a sanity
# check; C is tuned on validation in the classifier cell.

[lstm] checks passed | features 256-d saved for ['train', 'val', 'test']
majority-class val acc = 0.500
model pool  val_acc  val_macro_f1
 lstm last    0.574         0.574
 lstm mean    0.626         0.626


[{'model': 'lstm',
  'pool': 'last',
  'val_acc': 0.574,
  'val_macro_f1': 0.57386193126573},
 {'model': 'lstm',
  'pool': 'mean',
  'val_acc': 0.6264,
  'val_macro_f1': 0.626338779345608}]

In [ ]:
# ==========================================
# CELL 4: PART II EMBEDDING VARIANTS (LSTM) — all settings x all seeds
# ==========================================
# Part II's LSTM differs from Part I's (untied output layer, padding_idx on the embedding),
# so the embedding comparison uses the Part II "random" model, not Part I's lstm.pt.
class Part2LSTM(nn.Module):
    """= LSTMLanguageModel from Part II Cell 7 (state_dict keys are identical)."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pad_id):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(hidden_dim, embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)
    def forward(self, x, hidden=None):
        out, hidden = self.lstm(self.drop(self.embedding(x)), hidden)
        return self.fc(self.proj(self.drop(out))), hidden

P2_SETTINGS = ["random", "own_sg", "public"]
P2_SEEDS = [42, 43, 44]

def load_part2(setting, seed):
    ckpt = torch.load(PART2_DIR / f"lstm_{setting}_s{seed}.pt", map_location="cpu", weights_only=False)
    assert ckpt["setting"] == setting and ckpt["seed"] == seed
    assert ckpt["token_to_id"] == token_to_id, "Part II vocabulary differs from Part I"
    cfg = ckpt["config"]
    m = Part2LSTM(len(vocabulary), cfg["embed_dim"], cfg["hidden_dim"], cfg["dropout"], PAD_ID)
    m.load_state_dict(ckpt["model_state_dict"])
    return freeze(m)

set_seed()
p2_rows = []
for setting in P2_SETTINGS:
    for seed in P2_SEEDS:
        name = f"p2_{setting}_s{seed}"
        model = load_part2(setting, seed)
        run_extraction(name, model, lstm_hidden, recurrent_head)
        p2_rows += [dict(r, setting=setting, seed=seed) for r in quick_check(name, verbose=False)]
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

p2_val = pd.DataFrame(p2_rows)
summary = p2_val.groupby(["setting", "pool"], sort=False)[["val_acc", "val_macro_f1"]].agg(["mean", "std"])
print("Validation sanity check, mean ± std over 3 LM seeds (C = 1.0, not tuned yet):")
print(summary.round(3).to_string())

[p2_random_s42] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_random_s43] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_random_s44] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_own_sg_s42] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_own_sg_s43] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_own_sg_s44] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_public_s42] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_public_s43] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_public_s44] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_public_ft_s42] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_public_ft_s43] checks passed | features 256-d saved for ['train', 'val', 'test']
[p2_public_ft_s44] checks passed | features 256-d saved for ['train', 'val', 'test']
Validat

In [ ]:
# ==========================================
# CELL 5: CLASSIFIER — embedding settings only (Part II LSTM, mean pooling)
# ==========================================
# Same classifier and procedure for every setting: StandardScaler + logistic regression,
# C chosen by validation macro-F1, then ONE evaluation on the test set.
from sklearn.feature_extraction.text import TfidfVectorizer

SETTINGS = {"random": "Random (trained)", "own_sg": "Own Skip-gram (frozen)", "public": "GloVe-100 (frozen)"}
SEEDS = [42, 43, 44]
C_GRID = [1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100]
PRED_DIR = OUT_DIR / "predictions"; PRED_DIR.mkdir(exist_ok=True)
y = {s: np.load(FEAT_DIR / f"labels_{s}.npy") for s in SPLITS}

def fit_eval(name, X, scale=True):
    best = None
    for C in C_GRID:
        lr = LogisticRegression(C=C, max_iter=3000)
        clf = make_pipeline(StandardScaler(), lr) if scale else lr
        clf.fit(X["train"], y["train"])
        f1 = f1_score(y["val"], clf.predict(X["val"]), average="macro")
        if best is None or f1 > best[0]:
            best = (f1, C, clf)
    _, C, clf = best
    prob = clf.predict_proba(X["test"])[:, 1]          # test touched once, with the chosen C
    np.save(PRED_DIR / f"{name}.npy", prob)            # used by the error analysis (Cell 6)
    pred = (prob >= 0.5).astype(int)
    if C in (C_GRID[0], C_GRID[-1]):
        print(f"  note: {name} chose C={C}, at the edge of C_GRID")
    return {"C": C, "test_acc": accuracy_score(y["test"], pred),
            "test_macro_f1": f1_score(y["test"], pred, average="macro")}

def lm_ppl(setting, seed):
    return json.loads((PART2_DIR / f"result_{setting}_s{seed}.json").read_text())["test_ppl"]

# ---- 1) the three embedding settings, every LM seed ----
rows = []
for setting in SETTINGS:
    for seed in SEEDS:
        name = f"p2_{setting}_s{seed}_mean"
        X = {s: np.load(FEAT_DIR / f"{name}_{s}.npy") for s in SPLITS}
        rows.append({"setting": setting, "seed": seed, "lm_test_ppl": lm_ppl(setting, seed), **fit_eval(name, X)})
runs = pd.DataFrame(rows)
runs.to_csv(OUT_DIR / "results_embeddings_per_seed.csv", index=False)

# ---- 2) summary table (Table 9b): mean ± std over the 3 LM seeds ----
ms = lambda s: f"{s.mean():.3f} ± {s.std(ddof=1):.3f}"
table = runs.groupby("setting", sort=False).agg(
    lm_test_ppl=("lm_test_ppl", lambda s: round(s.mean(), 1)),
    test_acc=("test_acc", ms), test_macro_f1=("test_macro_f1", ms))
table.index = table.index.map(SETTINGS)

# ---- 3) paired difference vs random: same seed = same LSTM init and batch order in Part II ----
per_seed = runs.pivot(index="seed", columns="setting", values="test_acc")
diff = {s: per_seed[s] - per_seed["random"] for s in SETTINGS if s != "random"}
table["acc_diff_vs_random"] = ["—"] + [f"{d.mean():+.3f} ± {d.std(ddof=1):.3f} ({int((d > 0).sum())}/3 seeds better)"
                                       for d in diff.values()]
table.to_csv(OUT_DIR / "results_embeddings_summary.csv")

# ---- 4) reference points (same test set, not LM representations) ----
texts = {s: pd.read_csv(TOK_DIR / f"{s}_kept.csv")[TEXT_COL].astype(str) for s in SPLITS}
tfidf = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False, ngram_range=(1, 2), min_df=2)
X_tfidf = {"train": tfidf.fit_transform(texts["train"])}
X_tfidf.update({s: tfidf.transform(texts[s]) for s in ("val", "test")})
tfidf_res = fit_eval("tfidf_bigram", X_tfidf, scale=False)
majority_acc = max(y["test"].mean(), 1 - y["test"].mean())

print(" LSTM, mean pooling, test set (mean ± std over 3 LM seeds):")
print(table.to_string())
print(f"\nReference: majority class acc = {majority_acc:.3f} | "
      f"TF-IDF bigram acc = {tfidf_res['test_acc']:.3f}, macro-F1 = {tfidf_res['test_macro_f1']:.3f}")
print("\nPer-seed test accuracy:")
print(per_seed[list(SETTINGS)].round(3).to_string())

# ---- 5) does LM ranking agree with downstream ranking? ----
avg = runs.groupby("setting")[["lm_test_ppl", "test_acc"]].mean()
print(f"\nRanking by LM perplexity (best first): {list(avg.sort_values('lm_test_ppl').index)}")
print(f"Ranking by test accuracy (best first): {list(avg.sort_values('test_acc', ascending=False).index)}")

 Part II LSTM, mean pooling, test set (mean ± std over 3 LM seeds):
                        lm_test_ppl       test_acc  test_macro_f1                 acc_diff_vs_random
setting                                                                                             
Random (trained)              141.2  0.624 ± 0.009  0.624 ± 0.009                                  —
Own Skip-gram (frozen)        141.7  0.635 ± 0.013  0.635 ± 0.013  +0.011 ± 0.019 (2/3 seeds better)
GloVe-100 (frozen)            137.5  0.670 ± 0.011  0.670 ± 0.011  +0.046 ± 0.003 (3/3 seeds better)

Reference: majority class acc = 0.500 | TF-IDF bigram acc = 0.759, macro-F1 = 0.759

Per-seed test accuracy:
setting  random  own_sg  public
seed                           
42        0.634   0.622   0.683
43        0.615   0.634   0.661
44        0.623   0.648   0.666

Ranking by LM perplexity (best first): ['public', 'random', 'own_sg']
Ranking by test accuracy (best first): ['public', 'own_sg', 'random']


In [ ]:
# ==========================================
# CELL 6: ERROR ANALYSIS — where do two representations fail, and is it systematic?
# ==========================================
from scipy.stats import binomtest

A, B = "p2_random_s42_mean", "p2_public_s42_mean"     # random vs frozen GloVe, same LSTM, mean pooling
test_df = pd.read_csv(TOK_DIR / "test_kept.csv")
tok_test = torch.load(TOK_DIR / "test.pt")
y_test = y["test"]
prob = {r: np.load(PRED_DIR / f"{r}.npy") for r in (A, B)}
pred = {r: (p >= 0.5).astype(int) for r, p in prob.items()}

# ---- 1) tag every test review with linguistic properties that plausibly cause errors ----
NEGATORS = {"not", "no", "never", "nothing", "neither", "nor", "without", "hardly", "barely"}
CONTRAST = {"but", "however", "although", "though", "yet", "despite", "while", "still"}
toks = test_df[TEXT_COL].astype(str).map(tokenize)
tags = pd.DataFrame({
    "negation":   toks.map(lambda ts: any(t in NEGATORS or t.endswith("n't") for t in ts)),
    "contrast":   toks.map(lambda ts: any(t in CONTRAST for t in ts)),
    "truncated":  toks.map(lambda ts: len(ts) + 1 > MAX_LEN),          # +1 for <s>
    "unk_heavy":  toks.map(lambda ts: np.mean([t not in token_to_id for t in ts]) >= 0.15),
    "very_short": toks.map(lambda ts: len(ts) <= 6),
})

rows = [{"subset": "all test reviews", "n": len(y_test), "share_%": 100.0,
         f"err_{A}_%": 100 * (pred[A] != y_test).mean(), f"err_{B}_%": 100 * (pred[B] != y_test).mean()}]
for t in tags.columns:
    m = tags[t].to_numpy()
    if m.sum() == 0:
        continue
    rows.append({"subset": t, "n": int(m.sum()), "share_%": 100 * m.mean(),
                 f"err_{A}_%": 100 * (pred[A][m] != y_test[m]).mean(),
                 f"err_{B}_%": 100 * (pred[B][m] != y_test[m]).mean()})
err_by_tag = pd.DataFrame(rows)
err_by_tag.to_csv(OUT_DIR / f"errors_by_tag_{A}_vs_{B}.csv", index=False)
print("Error rate by subset (a subset is a systematic weakness if its error rate is well above 'all'):")
print(err_by_tag.round(1).to_string(index=False))

# ---- 2) where do the two representations disagree? (McNemar's exact test) ----
ok_A, ok_B = pred[A] == y_test, pred[B] == y_test
n_A_only, n_B_only = int((ok_A & ~ok_B).sum()), int((~ok_A & ok_B).sum())
p_mcnemar = binomtest(n_A_only, n_A_only + n_B_only, 0.5).pvalue if n_A_only + n_B_only else 1.0
print(f"\nBoth right {int((ok_A & ok_B).sum())} | only {A} right {n_A_only} | only {B} right {n_B_only} "
      f"| both wrong {int((~ok_A & ~ok_B).sum())} | McNemar exact p = {p_mcnemar:.4f}")

# ---- 3) candidate examples to read (what the model actually saw: <unk>s, truncation) ----
def model_input(i):
    return " ".join(vocabulary[j] for j in tok_test["ids"][i, : tok_test["lengths"][i]].tolist()[1:])

table = test_df[[TEXT_COL]].copy()
table["label"] = y_test
table["model_input"] = [model_input(i) for i in range(len(table))]
table[f"p_pos_{A}"], table[f"p_pos_{B}"] = prob[A].round(3), prob[B].round(3)
table["tags"] = tags.apply(lambda r: ",".join(c for c in tags.columns if r[c]), axis=1)
table["case"] = np.select([ok_A & ok_B, ok_A & ~ok_B, ~ok_A & ok_B],
                          ["both right", f"only {A} right", f"only {B} right"], "both wrong")
table.to_csv(OUT_DIR / f"error_table_{A}_vs_{B}.csv", index=False)   # read this file to pick examples

def show(mask, conf_col, title, k=5):
    sub = table[mask].copy()
    sub["confidence"] = (sub[conf_col] - 0.5).abs()           # confidently wrong = most informative
    print(f"\n--- {title} ({mask.sum()} reviews; top {k} by confidence) ---")
    for _, r in sub.sort_values("confidence", ascending=False).head(k).iterrows():
        print(f"[label {'Fresh' if r.label else 'Rotten'} | p_pos {A}={r[f'p_pos_{A}']:.2f}, "
              f"{B}={r[f'p_pos_{B}']:.2f} | tags: {r.tags or '-'}]\n   {r[TEXT_COL][:200]}")

show(~ok_A & ~ok_B, f"p_pos_{A}", "Both wrong (shared failures / possible label noise)")
show(ok_A & ~ok_B, f"p_pos_{B}", f"Only {A} right")
show(~ok_A & ok_B, f"p_pos_{A}", f"Only {B} right")

# ---- 4) does the disagreement pattern hold for every LM seed? ----
for sd in [42, 43, 44]:
    a = np.load(PRED_DIR / f"p2_random_s{sd}_mean.npy") >= 0.5
    b = np.load(PRED_DIR / f"p2_public_s{sd}_mean.npy") >= 0.5
    oa, ob = a == y_test, b == y_test
    na, nb = int((oa & ~ob).sum()), int((~oa & ob).sum())
    print(f"seed {sd}: only random right {na} | only GloVe right {nb} | "
          f"McNemar p = {binomtest(na, na + nb, 0.5).pvalue:.4f}")

Error rate by subset (a subset is a systematic weakness if its error rate is well above 'all'):
          subset    n  share_%  err_p2_random_s42_mean_%  err_p2_public_s42_mean_%
all test reviews 2500    100.0                      36.6                      31.7
        negation  612     24.5                      39.7                      36.6
        contrast  683     27.3                      44.7                      39.8
       truncated  625     25.0                      38.1                      37.9
       unk_heavy  435     17.4                      38.2                      29.9
      very_short   67      2.7                      43.3                      37.3

Both right 1274 | only p2_random_s42_mean right 310 | only p2_public_s42_mean right 433 | both wrong 483 | McNemar exact p = 0.0000

--- Both wrong (shared failures / possible label noise) (483 reviews; top 5 by confidence) ---
[label Fresh | p_pos p2_random_s42_mean=0.12, p2_public_s42_mean=0.09 | tags: unk_heavy]
   Gr